# Bias figure — CanESM2‑WRF D03 vs. observations (1986–2005)

**Plot step only.** All numbers are precomputed by `make_bias_figure.py`
(compute step) and stored in `pickles/bias_figure.pkl` +
`pickles/bias_filter_comparison.csv`.  If you change data or conventions,
delete `pickles/bias_figure.pkl` and re-run the script first.

Layout (3 columns × 2 rows, one column per variable, single model = **D03**):

|  | Temperature (°C) | Precipitation (top: mm/day, bottom: mm/season) | Wind speed (m/s) |
|---|---|---|---|
| **Top**    | obs 1986–2005 climatology map (station scatter) | obs climatology map | obs climatology map |
| **Bottom** | model vs. obs (annual point + seasonal points in legend) | model vs. obs | model vs. obs |

* Stations: ECCC + BCH + NOAA land (incl. 5 USW WA/OR wind stations) + 5 D03
  wind buoys + 8 D03 temperature buoys (buoys drawn as diamonds).
* Two horizontal colorbars under each column: climatology scale (top) and
  bias scale (bottom).
* Bias = model − obs (degC for T; % for PR and wind).
* Gated annuals: season‑first pool, 90 % per‑season completeness gate,
  ≥15 surviving years (same as `seasonal_pooling.py`).  Raw annuals:
  simple 20‑yr mean/sum.  Section 3 compares the two per station.
* Bias colors/limits from Eva's `bias/spatial_maps.py`; climatology colormaps
  match the project's mean–change maps (RdBu_r / BrBG / PuOr).

Run top‑to‑bottom with the `py_2024` environment (Python 3.9).


## 1. Setup

In [1]:
import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cf

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE      = os.path.dirname(os.path.abspath('.'))
BASE      = os.getcwd()                      # run from the notebook dir
DOMAIN_DIR = os.path.join(BASE, 'plotting_files')
REPO_DIR   = os.path.join(BASE, 'WRF-CanESM2')
OUT_DIR    = os.path.join(BASE, 'figures', 'bias')
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, REPO_DIR)
import WRFDomainLib

FONT = 16
SEASONS = ('DJF', 'MAM', 'JJA', 'SON')       # legend order, winter first
VARS = ('t', 'pr', 'wind')
VAR_UNITS = {'t': '°C', 'pr': 'mm/yr', 'wind': 'm/s'}   # annual quantities
# panel-specific units: top row (climatology) pr in mm/day;
# bottom row (model vs obs) pr in mm/season for the seasonal points
VAR_UNITS_CLIM = {'t': '°C', 'pr': 'mm/day', 'wind': 'm/s'}
VAR_UNITS_SCAT = {'t': '°C', 'pr': 'mm/season', 'wind': 'm/s'}
VAR_TITLE = {'t': 'Temperature', 'pr': 'Precipitation', 'wind': 'Wind speed'}
PANEL_LETTER = {'t': 'a', 'pr': 'b', 'wind': 'c'}          # top row (maps)
PANEL_LETTER_SCAT = {'t': 'd', 'pr': 'e', 'wind': 'f'}   # bottom row (scatters)
PANEL_YMAX={'t':26 , 'pr': 1500, 'wind': 10} 
PANEL_YMIN={'t':-15 , 'pr': -5, 'wind': 0} 
# ---- load the precomputed pickle (make_bias_figure.py) ----
PKL = os.path.join(BASE, 'pickles', 'bias_figure.pkl')
with open(PKL, 'rb') as f:
    D = pickle.load(f)
print('pickle generated:', D['_generated'])

LAT, LON, ELEV = D['stations']['lat'], D['stations']['lon'], D['stations']['elev']
RAW = {v: (o, m) for v, (o, m) in D['raw'].items()}          # raw period mean (scalar/station)
RAW_ANN = {v: (o, m) for v, (o, m) in D['raw_ann'].items()}  # raw 20-annual arrays
GATED = {v: (o, m) for v, (o, m) in D['gated'].items()}      # gated annuals
SEAS = {v: D['seasonal'][v] for v in VARS}                   # mod/obs/bias frames
SKILL = D['skill']                                           # Eva skill biases

# D03 buoys: drawn with a distinct marker (diamond); land = dot
BUOYS = set(D['stations']['buoy_ids'])
_is_buoy = pd.Series([s in BUOYS for s in LAT.index], index=LAT.index)
print(f'{len(LAT)} stations ({len(BUOYS)} buoys) | generated {D["_generated"]}')

# ---- gridded 1986-2005 D03 climatology (make_climo_d03.py) ----
CLIMO_PKL = os.path.join(BASE, 'pickles', 'climo_d03_hist.pkl')
with open(CLIMO_PKL, 'rb') as f:
    CLIMO = pickle.load(f)['out']
print('gridded climatology loaded: %s' %
      {v: CLIMO[v]['clim'].shape for v in CLIMO})


pickle generated: 2026-09-30T04:04:54
181 stations (8 buoys) | generated 2026-09-30T04:04:54
gridded climatology loaded: {'t': (300, 300), 'pr': (300, 300), 'wind': (300, 300)}


ERROR 1: PROJ: proj_create_from_database: Open of /home/amh001/space_fs7/software_2022/python/py_2024/share/proj failed


## 2. Map base (D02/D03 terrain, D03 box, Lambert projection)

Same base as `plot_elev_bias_revised.py` / Eva's `bias/spatial_maps.py`.


In [2]:
def read_geo_em(fname):
    from netCDF4 import Dataset
    nc = Dataset(os.path.join(DOMAIN_DIR, fname))
    lat, lon, hgt = (nc[v][:] for v in ('XLAT_C', 'XLONG_C', 'HGT_M'))
    nc.close()
    return np.squeeze(lat), np.squeeze(lon), np.squeeze(hgt)

la3, lo3, ht3 = read_geo_em('geo_em.d03.nc')
la2, lo2, ht2 = read_geo_em('geo_em.d02.nc')

# local bathymetry grid (plotting_files/new_bath.nc) - offline ocean fill
from netCDF4 import Dataset as _DS
with _DS(os.path.join(DOMAIN_DIR, 'new_bath.nc')) as _b:
    BATH_LON = _b['lon'][:]
    BATH_LAT = _b['lat'][:]
    BATH = _b['elevation'][:]
_BATH_OCEAN = np.ma.masked_where(BATH > 0, BATH)
print('bathymetry loaded: %s (ocean cells: %d)' %
      (BATH.shape, int(_BATH_OCEAN.count())))

WPS = os.path.join(DOMAIN_DIR, 'namelist.wps.txt')
wpsproj, latlonproj, corner_lat_full, corner_lon_full, length_x, length_y = \
    WRFDomainLib.calc_wps_domain_info(WPS)
corner_x3, corner_y3 = WRFDomainLib.reproject_corners(
    corner_lon_full[2, :], corner_lat_full[2, :], wpsproj, latlonproj)

def new_map_ax(ax):
    """Draw terrain base + D03 box into an existing (Lambert) axis."""
    ax.pcolormesh(lo2, la2, ht2, cmap='terrain', vmin=0, vmax=3000,
                  alpha=0.30, transform=ccrs.PlateCarree(), zorder=0)
    ax.pcolormesh(lo3, la3, ht3, cmap='terrain', vmin=0, vmax=3000,
                  alpha=0.40, transform=ccrs.PlateCarree(), zorder=0)
    rxf = corner_x3[0] / 65
    ryf = -corner_y3[0] / 13
    ax.add_patch(mpl.patches.Rectangle(
        (corner_x3[0] + rxf, corner_y3[0] + ryf), length_x[2], length_y[2],
        fill=None, lw=1.5, edgecolor='red', zorder=2))
    ax.text(-3700000, 700000, 'D03', va='top', ha='left', fontweight='bold',
            size=FONT, color='red', zorder=2)
    ax.set_extent([-131, -119, 46, 52], crs=ccrs.PlateCarree())
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linestyle='--', alpha=0.7)
    gl.top_labels = True; gl.right_labels = True
    gl.bottom_labels = False; gl.left_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180, -49, 4))
    gl.ylocator = mticker.FixedLocator(np.arange(0, 81, 4))
    gl.label_style = {'inline': False}
    for lbl in gl.label_artists:
        lbl.set_fontsize(FONT - 2)
    ax.tick_params(labelsize=FONT - 2)
    return ax

def new_map_ax_plain(ax):
    """Same frame as new_map_ax but without the terrain pcolormesh:
    white background, D03 box + gridlines only (used for the climatology
    row where terrain would compete with the station dots)."""
    rxf = corner_x3[0] / 65
    ryf = -corner_y3[0] / 13
    ax.add_patch(mpl.patches.Rectangle(
        (corner_x3[0] + rxf, corner_x3[0] + ryf), length_x[2], length_y[2],
        fill=None, lw=1.5, edgecolor='red', zorder=2))
    ax.text(-3700000, 700000, 'D03', va='top', ha='left', fontweight='bold',
            size=FONT, color='red', zorder=2)
    ax.set_extent([-131, -119, 46, 52], crs=ccrs.PlateCarree())
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linestyle='--', alpha=0.7)
    gl.top_labels = True; gl.right_labels = True
    gl.bottom_labels = False; gl.left_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180, -49, 4))
    gl.ylocator = mticker.FixedLocator(np.arange(0, 81, 4))
    gl.label_style = {'inline': False}
    for lbl in gl.label_artists:
        lbl.set_fontsize(FONT - 2)
    ax.tick_params(labelsize=FONT - 2)
    #ax.set_facecolor('white')
    return ax

# colormaps & limits (Eva's bias/spatial_maps.py)
COLORS_T   = ['#142f60', '#3465aa', '#5392c1', '#99c4dd', '#d3e5f0', '#f7f7f7',
              '#fadcc8', '#eea785', '#ce6451', '#ab242f', '#630921']
COLORS_PR  = ['#386158', '#41847e', '#67aba5', '#a0d6cd', '#d5edea', '#f7f7f7',
              '#f3e7c4', '#dbc37f', '#b88234', '#865214', '#503009'][::-1]
COLORS_WSP = ['#424c03', '#41641a', '#4b8c49', '#79b17d', '#aed0b2', '#d7e3e0',
              '#aec3d5', '#7394b5', '#3e6896', '#294072', '#2c194d'][::-1]
CMAP_T = mpl.colors.LinearSegmentedColormap.from_list(
'temp_bias', COLORS_T, N=256
)
CMAP_PR = mpl.colors.LinearSegmentedColormap.from_list(
'precip_bias', COLORS_PR, N=256
)
CMAP_WSP = mpl.colors.LinearSegmentedColormap.from_list(
'wind_bias', COLORS_WSP, N=256
)
VAR_CMAP = {
't': ('RdBu_r', -8, 8),
'pr': ('BrBG',-200, 200),
'wind': ('PuOr',-200, 200),
}

# ---- obs climatology colormaps: match the project's mean-change maps ----
# (WRF-CanESM2/mean_changes: t = RdBu_r, pr = BrBG, wind = PuOr)
# ---- model climatology colormaps ----
CLIMO_T = [
    '#383593', '#4f74b2', '#7dacd0', '#b1d8e8', '#e2f3f8', '#fdffc1',
    '#fae194', '#f6b066', '#eb7249', '#ce3a2f', '#9e152a'
]

CLIMO_PR = [
    '#f2ffda', '#ecf8b3', '#c9e9b5', '#88ccbb', '#56b4c3',
    '#3c8fbe', '#335da6', '#2e3392', '#121c57'
]

CLIMO_WSP = [
    '#f2ffda', '#ecf8b3', '#c9e9b5', '#88ccbb', '#56b4c3',
    '#3c8fbe', '#335da6', '#2e3392', '#121c57'
]

CLIMO_CMAP = {
    't': (
        mpl.colors.LinearSegmentedColormap.from_list('climo_t', CLIMO_T, N=256),
        -12, 18
    ),
    'pr': (
        mpl.colors.LinearSegmentedColormap.from_list('climo_pr', CLIMO_PR, N=256),
        0, 12
    ),
    'wind': (
        mpl.colors.LinearSegmentedColormap.from_list('climo_wind', CLIMO_WSP, N=256),
        0, 12
    ),}

print('map base ready')


bathymetry loaded: (1021, 715) (ocean cells: 272635)
map base ready


## 3. Two ways of filtering: season‑first gate vs. raw 20‑yr mean

The annual bias can be computed two ways:

* **Gated (skill)** — obs pooled day → season → year with the 90 %
  per‑season completeness gate, year dropped if any season missing,
  ≥15 surviving years; model restricted to the same surviving years.
* **Raw (taylor)** — simple mean/sum of all 20 annual values, no gate.

Per‑station comparison table (also saved to
`pickles/bias_filter_comparison.csv`) and a 3‑panel scatter.


In [3]:
def _bias(o, m, var):
    return 100.0 * (m - o) / o if var == 'pr' else m - o

cmp_frames = []
for var in VARS:
    o_g, m_g = GATED[var]
    o_raw, m_raw = RAW[var]
    b_g = _bias(o_g, m_g, var)
    b_r = _bias(o_raw.reindex(o_g.index), m_raw.reindex(o_g.index), var)
    d = b_g - b_r
    both = b_g.notna() & b_r.notna()
    cmp_frames.append(pd.DataFrame({
        'var': var, 'station': o_g.index,
        'bias_gated': b_g.values, 'bias_raw': b_r.values}))
    print(f"{var}: {int(both.sum())} stations | "
          f"mean|diff|={d[both].abs().mean():.3f}  max|diff|={d[both].abs().max():.3f} | "
          f"gated mean bias={b_g[both].mean():.3f}  raw mean bias={b_r[both].mean():.3f}")

CMP = pd.concat(cmp_frames, ignore_index=True)
CMP.to_csv(os.path.join(BASE, 'pickles', 'bias_filter_comparison.csv'),
           index=False, float_format='%.4f')

fig, axes = plt.subplots(1, 3, figsize=(8, 4.2))
for ax, var in zip(axes, VARS):
    sub = CMP[CMP['var'] == var].dropna()
    bg, br = sub['bias_gated'].values, sub['bias_raw'].values
    ax.scatter(br, bg, s=22, c='#444444', edgecolor='k', linewidth=0.3, zorder=2)
    lim = float(max(np.abs(bg).max(), np.abs(br).max())) * 1.05
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, alpha=0.5, zorder=1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel('Raw (taylor) annual bias', fontsize=FONT)
    ax.set_ylabel('Gated (skill) annual bias', fontsize=FONT)
    ax.set_title(f'{VAR_TITLE[var]}\nn={len(sub)}', fontsize=FONT)
    ax.tick_params(labelsize=FONT - 1)
    ax.set_aspect('equal', adjustable='datalim')
fig.suptitle('Annual bias: season-first gate vs. raw 20-yr mean (per station, D03)',
             fontsize=FONT + 1, y=1.02)
fig.tight_layout()
out = os.path.join(OUT_DIR, 'd03_bias_filter_compare.png')
fig.savefig(out, dpi=300, bbox_inches='tight'); plt.close(fig)
print('saved', out)


t: 154 stations | mean|diff|=0.014  max|diff|=0.192 | gated mean bias=-1.040  raw mean bias=-1.036
pr: 159 stations | mean|diff|=0.594  max|diff|=6.730 | gated mean bias=-7.106  raw mean bias=-7.229
wind: 23 stations | mean|diff|=0.006  max|diff|=0.088 | gated mean bias=0.728  raw mean bias=0.722
saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/d03_bias_filter_compare.png


## 4. Top row — D03 model climatology maps (1986–2005)

Gridded 20-yr WRF D03 climatology (CF-compliant daily files → time mean),
plotted Eva-faithful (`plot_all_d03_climo`): pcolormesh with her exact
color lists/ranges (t: −12…18 °C; pr: 0…12 mm/day; wind: 0…12 m/s),
lightblue ocean, D03 box, gridlines.  Obs stations drawn on top as thin
black dots (buoys as small diamonds) to show the validation network.
Each panel carries its own pair of horizontal colorbars: climatology
scale (top) and bias scale (bottom, Eva's ListedColormaps).


In [4]:
def _station_bias_series(var):
    """Per-station model - obs bias on the figure's bias scale.

    Uses the GATED, like-for-like period-mean bias (obs pooled day->season
    with the 90% completeness gate, >=15 surviving years, model restricted to
    the same surviving years) - the SAME method as the skill tables & the
    validated buoy tables (the paper).  D['gated'] reproduces
    skill_seasonal.pkl['D03'] to ~0.

    t: °C
    pr, wind: percent

    Returns
    -------
    pandas.Series
        Bias indexed by station ID.
    """
    import pandas as _pd

    o, m = D['gated'][var]

    o = _pd.Series(o).astype(float)
    m = _pd.Series(m).astype(float)

    bias = m - o

    if var == 't':
        return bias

    return 100.0 * bias / o


def clim_panel(var):
    """Plot D03 climatology with station/buoy biases.

    The map shows the gridded 1986-2005 D03 climatology with station/buoy
    model-minus-observation biases overlaid. Two horizontal colorbars show
    the climatology and bias scales.
    """

    # ------------------------------------------------------------------
    # Data and colour scales
    # ------------------------------------------------------------------
    c = CLIMO[var]
    lat = c['lat']
    lon = c['lon']
    val = c['clim']

    clim_cmap, vmin_c, vmax_c = CLIMO_CMAP[var]
    bias_cmap, vmin_b, vmax_b = VAR_CMAP[var]

    # Station biases aligned with station locations
    sbias = _station_bias_series(var).reindex(LAT.index)
    okb = sbias.notna().reindex(LAT.index, fill_value=False)

    # ------------------------------------------------------------------
    # Figure
    # ------------------------------------------------------------------
    fig, ax = plt.subplots(
        subplot_kw={'projection': wpsproj},
        figsize=(6, 5.6)
    )

    # ------------------------------------------------------------------
    # Gridded climatology
    # ------------------------------------------------------------------
    ax.pcolormesh(
        lon,
        lat,
        val,
        cmap=clim_cmap,
        vmin=vmin_c,
        vmax=vmax_c,
        transform=ccrs.PlateCarree(),
        zorder=2
    )
    coastline = cf.COASTLINE.with_scale('10m')
    ax.add_feature(coastline, linewidth=0.2, edgecolor='black', zorder=3)


    # ------------------------------------------------------------------
    # Set bounds from the actual D03 climatology grid
    #
    # Transform the lat/lon grid into the native map projection and use
    # those coordinates to determine the displayed map extent.
    # ------------------------------------------------------------------
    xy = wpsproj.transform_points(
        ccrs.PlateCarree(),
        np.asarray(lon),
        np.asarray(lat)
    )

    x = xy[..., 0]
    y = xy[..., 1]

    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    ymin = np.nanmin(y)
    ymax = np.nanmax(y)

    # Crop very slightly inside the outer grid bounds so that the
    # pcolormesh fills the entire rectangular panel without white wedges.
    crop = 0.01

    dx = xmax - xmin
    dy = ymax - ymin

    ax.set_xlim(
        xmin + crop * dx,
        xmax - crop * dx
    )

    ax.set_ylim(
        ymin + crop * dy*0.95,
        ymax - crop * dy
    )

    # ------------------------------------------------------------------
    # Gridlines and labels
    # ------------------------------------------------------------------
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linestyle='--',
        alpha=0.7
    )

    gl.top_labels = True
    gl.right_labels = True
    gl.bottom_labels = False
    gl.left_labels = False

    gl.xlocator = mticker.FixedLocator(
        np.arange(-180, -49, 4)
    )
    gl.ylocator = mticker.FixedLocator(
        np.arange(0, 81, 4)
    )

    gl.label_style = {'inline': False}

    for lbl in gl.label_artists:
        lbl.set_fontsize(FONT - 2)

    # ------------------------------------------------------------------
    # Station/buoy bias
    # ------------------------------------------------------------------
    land = ~_is_buoy
    bu = _is_buoy

    # Land stations
    land_ok = land & okb

    if land_ok.any():
        ax.scatter(
            LON[land_ok],
            LAT[land_ok],
            c=sbias[land_ok],
            s=80,
            cmap=bias_cmap,
            vmin=vmin_b,
            vmax=vmax_b,
            edgecolor='k',
            linewidth=0.5,
            marker='o',
            zorder=4,
            transform=ccrs.PlateCarree()
        )

    # Buoys
    bu_ok = bu & okb

    if bu_ok.any():
        ax.scatter(
            LON[bu_ok],
            LAT[bu_ok],
            c=sbias[bu_ok],
            s=80,
            cmap=bias_cmap,
            vmin=vmin_b,
            vmax=vmax_b,
            edgecolor='k',
            linewidth=0.5,
            marker='D',
            zorder=5,
            transform=ccrs.PlateCarree()
        )

    # ------------------------------------------------------------------
    # Colorbars
    # ------------------------------------------------------------------
    unit = '%' if var in ('pr', 'wind') else '\N{DEGREE SIGN}C'

    smc = cm.ScalarMappable(
        cmap=clim_cmap,
        norm=mcolors.Normalize(vmin=vmin_c, vmax=vmax_c)
    )

    smb = cm.ScalarMappable(
        cmap=bias_cmap,
        norm=mcolors.Normalize(vmin=vmin_b, vmax=vmax_b)
    )

    # Positions of the two side-by-side colorbars
    x0 = 0.13
    x1 = 0.93
    gap = 0.04

    half = (x1 - x0 - gap) / 2.0

    yb = 0.050
    hb = 0.035

    cax_c = fig.add_axes([
        x0,
        yb,
        half,
        hb
    ])

    cax_b = fig.add_axes([
        x1 - half,
        yb,
        half,
        hb
    ])

    # Climatology ticks
    ticks_c = np.linspace(vmin_c, vmax_c, 4)

    cb_c = fig.colorbar(
        smc,
        cax=cax_c,
        orientation='horizontal',
        extend='both',
        ticks=ticks_c
    )
    cb_c.ax.set_xlabel( f'{VAR_TITLE[var]} ({VAR_UNITS_CLIM[var]})', rotation=0, fontsize=FONT, ha='center')
    #cb_c.ax.yaxis.label.set_position((0.5, -0.62))
    cb_c.ax.tick_params(
        labelsize=FONT - 1
    )

   # cb_c.ax.set_title(
    #    f'{VAR_TITLE[var]} ({VAR_UNITS_CLIM[var]})',
     #   fontsize=FONT,
      #  loc='bottom',
       # pad=8
    #)

    # Bias colorbar
    cb_b = fig.colorbar(
        smb,
        cax=cax_b,
        orientation='horizontal',
        extend='both',
        ticks=[
            vmin_b, 
            0,
            vmax_b
        ]
    )

    cb_b.ax.tick_params(
        labelsize=FONT - 1
    )
    cb_b.ax.set_xlabel(f'Bias ({unit})', rotation=0, fontsize=FONT, ha='center')


    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------
    # panel letter, inside the axes (upper-left)
    ax.text(0.03, 0.97, PANEL_LETTER[var], transform=ax.transAxes,
            ha='left', va='top', fontsize=FONT, fontweight='bold',
            color='black', zorder=10)

    out = os.path.join(
        OUT_DIR,
        f'd03_{var}_climatology_map.png'
    )

    fig.savefig(
        out,
        dpi=300,
        bbox_inches='tight'
    )

    plt.close(fig)

    print('saved', out)

    return out


MAP_PNLS = {
    var: clim_panel(var)
    for var in VARS
}

saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/d03_t_climatology_map.png
saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/d03_pr_climatology_map.png
saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/d03_wind_climatology_map.png


## 5. Bottom row — model vs. obs (annual point + seasonal points in legend)

Annual point from RAW annuals (mean of the 20 annual values per station;
pr in mm/yr).  Seasonal points from the gated seasonal model/obs in the
pickle (pr in mm/season); the per-season legend text reports the pooled
RMSE and R² of model vs. obs across stations for that season.

In [5]:

BLUE = '#2166ac'
# season -> marker spec: (marker, filled?)
SEASON_STYLE = {
    'DJF': dict(marker='^', filled=True,  ms=9),   # filled triangle
    'MAM': dict(marker='o', filled=False, ms=9),   # open circle
    'JJA': dict(marker='s', filled=False, ms=8),   # open square
    'SON': dict(marker='D', filled=True,  ms=9),   # filled diamond
}

def _seasonal_rmse_r2(so, sm_, sst, seas):
    """Pooled RMSE (units) and Pearson R^2 (r**2) of model vs obs across
    stations, one season (values from the `seas` column)."""
    from scipy.stats import pearsonr
    ov = so.loc[sst, seas].astype(float).values
    mv = sm_.loc[sst, seas].astype(float).values
    ok = np.isfinite(ov) & np.isfinite(mv)
    ov, mv = ov[ok], mv[ok]
    if len(ov) < 3:
        return np.nan, np.nan
    rmse = float(np.sqrt(np.mean((mv - ov) ** 2)))
    r = float(pearsonr(mv, ov)[0])
    r2 = r * r  # squared Pearson correlation, always >= 0
    return rmse, r2

def scatter_panel(var):
    so, sm_ = SEAS[var]['obs'], SEAS[var]['mod']
    stations = [s for s in so.index if s in set(LAT.index) and s in set(sm_.index)]

    # pooled (obs, model) arrays across all seasons -> 1:1 + OLS model fit
    xs, ys = [], []
    for seas in SEASONS:
        o = so.loc[stations, seas].astype(float).values
        m = sm_.loc[stations, seas].astype(float).values
        ok = np.isfinite(o) & np.isfinite(m)
        xs.append(o[ok]); ys.append(m[ok])
    xs = np.concatenate(xs); ys = np.concatenate(ys)

    # seasonal metrics (pooled across stations) for the legend
    seasm = {}
    for seas in SEASONS:
        sst = [s for s in stations
               if np.isfinite(so.loc[s, seas]) and np.isfinite(sm_.loc[s, seas])]
        seasm[seas] = (sst, _seasonal_rmse_r2(so, sm_, sst, seas) if sst else (np.nan, np.nan))

    fig, ax = plt.subplots(figsize=(5, 4.6))
    # Land and buoy stations share the SAME per-season marker/color; the
    # legend RMSE/R2 is pooled over ALL points (land + buoy) for the season.
    for seas in SEASONS:
        sst, (rmse, r2) = seasm[seas]
        if not sst:
            continue
        st = SEASON_STYLE[seas]
        kw = dict(edgecolor='k', linewidth=0.8, zorder=3)
        if st['filled']:
            kw.update(facecolor=BLUE)
        else:
            kw.update(facecolor='none')
        ax.scatter(so.loc[sst, seas], sm_.loc[sst, seas],
                   marker=st['marker'], s=st['ms'] ** 2, **kw,alpha=0.6,
                   label=f'{seas}  (RMSE={rmse:.1f}, R²={r2:.2f})')
        

    # 1:1 reference line (dashed grey) + OLS model fit (solid black)
    lo = min(xs.min(), ys.min()); hi = max(xs.max(), ys.max())

    lo = lo - 0.03 * (hi - lo); hi = hi + 0.03 * (hi - lo)
    ax.plot([lo, hi], [lo, hi], color='0.55', ls='--', lw=1.0, zorder=1)
    b, a = np.polyfit(xs, ys, 1)
    xx = np.array([lo, hi])
    ax.plot(xx, a + b * xx, color='0.15', ls='-', lw=1.4, zorder=2)
           # label=f'Model fit (slope={b:.2f})')
    ax.set_ylim([PANEL_YMIN[var], PANEL_YMAX[var]]); ax.set_xlim([PANEL_YMIN[var], PANEL_YMAX[var]])
    plt.locator_params(axis='both', nbins=5) 

   # print('hi', np.round(hi), 'lo', np.round(lo))
    unit = {'t': '°C', 'pr': 'mm/season', 'wind': 'm/s'}[var]
    ax.set_xlabel(f'Obs ({unit})', fontsize=FONT)
    ax.set_ylabel(f'Model ({unit})', fontsize=FONT)
   # ax.set_title(f'D03 {VAR_TITLE[var]}: model vs. obs (seasonal)',
    #             fontsize=FONT + 1, pad=6)
    ax.tick_params(labelsize=FONT - 1)
    ax.legend(loc=(-0.02,0.8),fontsize=FONT - 3, framealpha=0.9)
    ax.set_aspect('equal', adjustable='datalim')

    # panel letter, inside the axes (lower-right)
    ax.text(0.97, 0.03, PANEL_LETTER_SCAT[var], transform=ax.transAxes,
            ha='right', va='bottom', fontsize=FONT, fontweight='bold',
            color='black', zorder=10)

    out = os.path.join(OUT_DIR, f'd03_{var}_model_vs_obs.png')
    fig.savefig(out, dpi=300, bbox_inches='tight'); plt.close(fig)
    print('saved', out)
    return out

SCAT_PNLS = {var: scatter_panel(var) for var in VARS}


saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/d03_t_model_vs_obs.png
saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/d03_pr_model_vs_obs.png
saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/d03_wind_model_vs_obs.png


## 6. Stitch into the final figure

Three columns (t / pr / wind) × two rows (climatology map on top, model–vs–obs
below); each column carries its own pair of horizontal colorbars (climatology
scale + bias scale) baked into the panel.
→ `bias.png` in the workspace root and `figures/bias/bias_figure.png`.


In [6]:
def stitch(out_path):
    from PIL import Image, ImageDraw
    cols = list(VARS)
    maps  = [Image.open(MAP_PNLS[v]).convert('RGB') for v in cols]
    scats = [Image.open(SCAT_PNLS[v]).convert('RGB') for v in cols]
    W = max(im.width for im in maps + scats)
    def norm(im, w=W):
        r = w / im.width
        return im.resize((w, int(im.height * r)), Image.LANCZOS)
    maps  = [norm(m) for m in maps]
    scats = [norm(s) for s in scats]
    gap, titles_h = 10, 46
    map_h = max(m.height for m in maps)
    sca_h = max(s.height for s in scats)
    canvas = Image.new('RGB', (W * 3 + gap * 4, titles_h + map_h + sca_h + gap * 3), 'white')
    dr = ImageDraw.Draw(canvas)
    # (panel letters a-f are drawn inside each panel by clim_panel/scatter_panel)

    y0 = titles_h
    for i, m in enumerate(maps):
        x = gap + i * (W + gap)
        canvas.paste(m, (x, y0 + (map_h - m.height) // 2))
    y1 = titles_h + map_h + gap
    for i, s in enumerate(scats):
        x = gap + i * (W + gap)
        canvas.paste(s, (x, y1 + (sca_h - s.height) // 2))
    canvas.save(out_path)
    print('saved', out_path, canvas.size)

stitch(os.path.join(BASE, 'bias.png'))
stitch(os.path.join(OUT_DIR, 'bias_figure.png'))


saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/bias.png (4684, 3248)
saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/bias/bias_figure.png (4684, 3248)
